# Notebook 04: Build FAISS Vector Store & Semantic RAG Search
This notebook constructs local FAISS `IndexFlatIP` indices for business and review vector spaces and executes semantic search queries with metadata filtering.

In [1]:
import pickle
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
FAISS_DIR = PROJECT_ROOT / 'models' / 'faiss_index'

with open(FAISS_DIR / "biz_embeddings.pkl", "rb") as f:
    biz_data = pickle.load(f)

with open(FAISS_DIR / "rev_embeddings.pkl", "rb") as f:
    rev_data = pickle.load(f)

biz_matrix, biz_meta = biz_data["matrix"], biz_data["meta"]
rev_matrix, rev_meta = rev_data["matrix"], rev_data["meta"]

dim = biz_matrix.shape[1]

# Initialize FAISS Inner Product (Cosine Similarity) Indices
biz_index = faiss.IndexFlatIP(dim)
biz_index.add(biz_matrix)

rev_index = faiss.IndexFlatIP(dim)
rev_index.add(rev_matrix)

print("=======================================================")
print("               FAISS INDEX BUILD METRICS               ")
print("=======================================================")
print(f"Business Index Total Vectors: {biz_index.ntotal:,}")
print(f"Review Index Total Vectors:   {rev_index.ntotal:,}")
print(f"Vector Dimensions:            {dim}")


               FAISS INDEX BUILD METRICS               
Business Index Total Vectors: 2,000
Review Index Total Vectors:   2,000
Vector Dimensions:            384


In [2]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def search_rag(query_text, doc_type="all", city=None, min_stars=None, top_k=3):
    query_vec = model.encode([query_text], normalize_embeddings=True).astype(np.float32)
    results = []

    if doc_type in ["all", "business"]:
        scores, indices = biz_index.search(query_vec, top_k * 3)
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(biz_meta): continue
            m = biz_meta[idx]
            if city and m.get("city", "").lower() != city.lower(): continue
            if min_stars and m.get("business_rating", 0) < min_stars: continue
            results.append({
                "score": float(score), "type": "business", "name": m.get("business_name"),
                "city": m.get("city"), "rating": m.get("business_rating"), "text": m.get("document_text")
            })

    if doc_type in ["all", "review"]:
        scores, indices = rev_index.search(query_vec, top_k * 3)
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(rev_meta): continue
            m = rev_meta[idx]
            if city and m.get("city", "").lower() != city.lower(): continue
            if min_stars and m.get("stars", 0) < min_stars: continue
            results.append({
                "score": float(score), "type": "review", "name": m.get("business_name"),
                "city": m.get("city"), "rating": m.get("stars"), "text": m.get("document_text")
            })

    results = sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]
    return results

print("[OK] Search function initialized.")


[OK] Search function initialized.


In [3]:
query_1 = "Top rated Italian restaurants with delicious pasta and outdoor seating"
print(f"--> QUERY 1: '{query_1}'\n")

res1 = search_rag(query_1, doc_type="all", top_k=3)
for i, r in enumerate(res1, 1):
    print(f"[{i}] Score: {r['score']:.4f} | Type: {r['type'].upper()} | Name: {r['name']} ({r['city']}) | Rating: {r['rating']} ⭐")
    print("    Text Excerpt:")
    snippet = r['text'].replace('\n', ' ')
    print(f"    {snippet[:180]}...")
    print("-" * 75)


[ERROR]: f-string expression part cannot include a backslash (<string>, line 8)

In [4]:
query_2 = "Great pizza place with fast service and friendly staff"
print(f"--> QUERY 2: '{query_2}'\n")

res2 = search_rag(query_2, doc_type="review", top_k=3)
for i, r in enumerate(res2, 1):
    print(f"[{i}] Score: {r['score']:.4f} | Type: {r['type'].upper()} | Name: {r['name']} ({r['city']}) | Rating: {r['rating']} ⭐")
    print("    Text Excerpt:")
    snippet = r['text'].replace('\n', ' ')
    print(f"    {snippet[:180]}...")
    print("-" * 75)


[ERROR]: f-string expression part cannot include a backslash (<string>, line 8)